# VTA 2025 Ridership Dashboard - GPU Accelerated (NVIDIA RAPIDS + Plotly Dash)

## Story: *"Optimizing VTA Transit: Where, When & How Silicon Valley Rides"*

This notebook is the **GPU-accelerated version** of the CPU dashboard.  
Key difference: **Data cleaning & aggregation use `cuDF` (NVIDIA RAPIDS)** instead of pandas, running on the GPU for massive speedup on 388K+ rows.

**Requirements:** NVIDIA GPU + CUDA + `pip install cudf-cu12 cuml-cu12`  
(Run on a machine with NVIDIA GPU, e.g., SJSU HPC, Google Colab with GPU, or your NVIDIA laptop)

---
**Steps:** GPU Data Cleaning (cuDF) → GPU Aggregation → Plotly Dash Dashboard

In [1]:
# Install dependencies (run once on GPU machine)
!pip install cudf-cu12 cuml-cu12 pandas openpyxl plotly dash dash-bootstrap-components numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 12.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cuda-nvcc-cu12
    Found existing installation: nvidia-cuda-nvcc-cu12 12.5.82
    Uninstalling nvidia-cuda-nvcc-cu12-12.5.82:
      Successfully uninstalled nvidia-cuda-nvcc-cu12-12.5.82


## Step 1: Load Data (pandas for Excel, then transfer to GPU)

In [2]:
import pandas as pd
import numpy as np
import cudf  # NVIDIA RAPIDS GPU DataFrame
import time
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Path to the dataset in Google Drive
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/DATA 230/Group Project/OCT_2025_RBS_FULL_DATA_SET.XLSX"

# cuDF cannot read Excel directly, so load with pandas first then move to GPU
print("Loading Excel with pandas ...")
start = time.time()
df_pd = pd.read_excel(DATA_PATH, engine="openpyxl")
load_time = time.time() - start
print(f"  Loaded in {load_time:.1f}s  |  Shape: {df_pd.shape}")

# Fix columns with datetime.time objects — cuDF cannot handle them directly
# Convert TRIP_TIME to fractional hours (numeric) before GPU transfer
if "TRIP_TIME" in df_pd.columns:
    def time_to_fraction(val):
        """Convert datetime.time or numeric to fraction of day."""
        import datetime
        if isinstance(val, datetime.time):
            return (val.hour + val.minute / 60 + val.second / 3600) / 24
        try:
            return float(val)
        except (TypeError, ValueError):
            return np.nan
    df_pd["TRIP_TIME"] = df_pd["TRIP_TIME"].apply(time_to_fraction)
    print("  Converted TRIP_TIME to numeric (fraction of day).")

# Convert any remaining object columns with mixed types to string
for col in df_pd.columns:
    if df_pd[col].dtype == "object":
        df_pd[col] = df_pd[col].astype(str)

# Transfer to GPU
print("Transferring to GPU (cuDF) ...")
start_gpu = time.time()
gdf = cudf.DataFrame(df_pd)
transfer_time = time.time() - start_gpu
print(f"  Transferred to GPU in {transfer_time:.2f}s")
print(f"  GPU DataFrame shape: {gdf.shape}")
print(f"  GPU memory used: {gdf.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Mounted at /content/drive
Loading Excel with pandas ...
  Loaded in 119.4s  |  Shape: (388202, 32)
  Converted TRIP_TIME to numeric (fraction of day).
Transferring to GPU (cuDF) ...
  Transferred to GPU in 1.68s
  GPU DataFrame shape: (388202, 32)
  GPU memory used: 133.6 MB


## Step 2: GPU-Accelerated Data Cleaning (cuDF)

All cleaning operations run on the GPU using cuDF - same logic as the CPU version but significantly faster on 388K+ rows.

In [3]:
start_clean_gpu = time.time()

# --- 2a. Drop fully empty rows & columns ---
before = gdf.shape
gdf = gdf.dropna(how="all")
gdf = gdf.dropna(axis=1, how="all")
print(f"2a. Dropped all-NaN rows/cols: {before} -> {gdf.shape}")

# --- 2b. Fix numeric columns on GPU ---
numeric_cols = [
    "BOARDINGS", "ALIGHTINGS", "TRIPS",
    "AVG_BOARDINGS", "AVG_ALIGHTINGS", "AVG_ACTIVITY",
    "PASS_LOAD", "PEAK_LOAD", "AVG_PEAK_LOAD",
    "SORT_ORDER", "STOP_ID", "TOTAL_SORT", "SORT_SP",
]
for col in numeric_cols:
    if col in gdf.columns:
        gdf[col] = cudf.to_numeric(gdf[col], errors="coerce")
print("2b. Numeric columns coerced (GPU).")

# --- 2c. Clean string columns on GPU ---
str_cols = [
    "ROUTE_NAME", "ROUTE_NUMBER", "SERVICE_PERIOD", "SERVICE_CODE",
    "DIRECTION_NAME", "BRANCH", "MAIN_CROSS_STREET", "CITY",
    "STOP_DISPLAY", "Additional_Notes", "PATTERN_KEY", "BLOCK",
]
for col in str_cols:
    if col in gdf.columns:
        gdf[col] = gdf[col].astype("str").str.strip()
        gdf[col] = gdf[col].replace({"nan": None, "None": None, "": None})
print("2c. String columns cleaned (GPU).")

# --- 2d. Convert TRIP_TIME (fraction of day) to hour on GPU ---
gdf["TRIP_TIME_NUMERIC"] = cudf.to_numeric(gdf["TRIP_TIME"], errors="coerce")
gdf["TRIP_HOUR"] = (gdf["TRIP_TIME_NUMERIC"] * 24).round(0).astype("Int64").clip(0, 23)
print("2d. TRIP_TIME -> TRIP_HOUR extracted (GPU).")

# --- 2e. Fill missing values on GPU ---
gdf["CITY"] = gdf["CITY"].fillna("Unknown")
gdf["Additional_Notes"] = gdf["Additional_Notes"].fillna("None")
for col in ["BOARDINGS", "ALIGHTINGS", "AVG_BOARDINGS", "AVG_ALIGHTINGS", "AVG_ACTIVITY"]:
    if col in gdf.columns:
        gdf[col] = gdf[col].fillna(0)
print("2e. Missing values filled (GPU).")

# --- 2f. Derive ROUTE_TYPE using GPU string matching ---
# Fill SERVICE_CODE NAs before string matching (cuDF str.contains doesn't support na param)
gdf["SERVICE_CODE"] = gdf["SERVICE_CODE"].fillna("")
sc_lower = gdf["SERVICE_CODE"].str.lower()
gdf["ROUTE_TYPE"] = "Other"
gdf["ROUTE_TYPE"] = gdf["ROUTE_TYPE"].where(~sc_lower.str.contains("express"), "Express")
gdf["ROUTE_TYPE"] = gdf["ROUTE_TYPE"].where(~sc_lower.str.contains("frequent"), "Frequent")
gdf["ROUTE_TYPE"] = gdf["ROUTE_TYPE"].where(~sc_lower.str.contains("local"), "Local")
gdf["ROUTE_TYPE"] = gdf["ROUTE_TYPE"].where(~sc_lower.str.contains("light rail"), "Light Rail")
gdf["ROUTE_TYPE"] = gdf["ROUTE_TYPE"].where(~sc_lower.str.contains("rapid"), "Rapid")
print(f"2f. ROUTE_TYPE distribution (GPU):\n{gdf['ROUTE_TYPE'].value_counts().to_pandas()}")

# --- 2g. Derive TIME_PERIOD using GPU conditional logic ---
hour = gdf["TRIP_HOUR"]
gdf["TIME_PERIOD"] = "Unknown"
gdf.loc[(hour >= 5) & (hour < 9), "TIME_PERIOD"] = "AM Peak (5-9)"
gdf.loc[(hour >= 9) & (hour < 15), "TIME_PERIOD"] = "Midday (9-3)"
gdf.loc[(hour >= 15) & (hour < 19), "TIME_PERIOD"] = "PM Peak (3-7)"
gdf.loc[(hour >= 19) & (hour < 23), "TIME_PERIOD"] = "Evening (7-11)"
gdf.loc[(hour >= 23) | (hour < 5), "TIME_PERIOD"] = "Late Night (11-5)"
print(f"2g. TIME_PERIOD distribution (GPU):\n{gdf['TIME_PERIOD'].value_counts().to_pandas()}")

# --- 2h. Derive TOTAL_ACTIVITY on GPU ---
gdf["TOTAL_ACTIVITY"] = gdf["BOARDINGS"] + gdf["ALIGHTINGS"]
print("2h. TOTAL_ACTIVITY created (GPU).")

# --- 2i. Drop duplicates on GPU ---
dupes = gdf.shape[0]
gdf = gdf.drop_duplicates()
dupes_removed = dupes - gdf.shape[0]
print(f"2i. Removed {dupes_removed} duplicate rows (GPU).")

clean_time_gpu = time.time() - start_clean_gpu
print(f"\n=== GPU Cleaning complete in {clean_time_gpu:.2f}s | Final shape: {gdf.shape} ===")

2a. Dropped all-NaN rows/cols: (388202, 32) -> (388202, 32)
2b. Numeric columns coerced (GPU).
2c. String columns cleaned (GPU).
2d. TRIP_TIME -> TRIP_HOUR extracted (GPU).
2e. Missing values filled (GPU).
2f. ROUTE_TYPE distribution (GPU):
ROUTE_TYPE
Frequent      254949
Local          79230
Light Rail     24646
Rapid          22006
Other           6407
Express          964
Name: count, dtype: int64
2g. TIME_PERIOD distribution (GPU):
TIME_PERIOD
Midday (9-3)         151198
PM Peak (3-7)         99629
AM Peak (5-9)         66601
Evening (7-11)        55104
Late Night (11-5)     10188
Unknown                5482
Name: count, dtype: int64
2h. TOTAL_ACTIVITY created (GPU).
2i. Removed 5481 duplicate rows (GPU).

=== GPU Cleaning complete in 1.28s | Final shape: (382721, 37) ===


## Step 3: GPU-Accelerated Aggregation (cuDF) + CPU vs GPU Benchmark

In [4]:
start_agg_gpu = time.time()

# --- Agg 1: Top 20 busiest stops ---
agg_stops_gdf = (
    gdf.groupby(["STOP_ID", "MAIN_CROSS_STREET", "CITY"])
    .agg({"BOARDINGS": "sum", "ALIGHTINGS": "sum",
          "TOTAL_ACTIVITY": "sum", "PEAK_LOAD": "mean"})
    .reset_index()
    .sort_values("BOARDINGS", ascending=False)
)
agg_stops_gdf.columns = ["STOP_ID", "MAIN_CROSS_STREET", "CITY",
                          "Total_Boardings", "Total_Alightings",
                          "Total_Activity", "Avg_Peak_Load"]
top_stops = agg_stops_gdf.head(20).to_pandas()
print(f"Agg 1: Top stops done (GPU)")

# --- Agg 2: Ridership by City ---
agg_city_gdf = (
    gdf.groupby("CITY")
    .agg({"BOARDINGS": "sum", "ALIGHTINGS": "sum",
          "STOP_ID": "nunique", "ROUTE_NUMBER": "nunique"})
    .reset_index()
    .sort_values("BOARDINGS", ascending=False)
)
agg_city_gdf.columns = ["CITY", "Total_Boardings", "Total_Alightings",
                         "Num_Stops", "Num_Routes"]
agg_city = agg_city_gdf.to_pandas()
print(f"Agg 2: City ridership done (GPU)")

# --- Agg 3: Ridership by Route ---
agg_route_gdf = (
    gdf.groupby(["ROUTE_NUMBER", "ROUTE_NAME", "ROUTE_TYPE"])
    .agg({"BOARDINGS": "sum", "ALIGHTINGS": "sum",
          "PEAK_LOAD": "mean", "PASS_LOAD": "mean", "TRIPS": "sum"})
    .reset_index()
    .sort_values("BOARDINGS", ascending=False)
)
agg_route_gdf.columns = ["ROUTE_NUMBER", "ROUTE_NAME", "ROUTE_TYPE",
                          "Total_Boardings", "Total_Alightings",
                          "Avg_Peak_Load", "Avg_Pass_Load", "Num_Trips"]
agg_route = agg_route_gdf.to_pandas()
print(f"Agg 3: Route ridership done (GPU)")

# --- Agg 4: Hourly ridership pattern ---
agg_hourly_gdf = (
    gdf.groupby(["TRIP_HOUR", "SERVICE_PERIOD"])
    .agg({"BOARDINGS": "sum"})
    .reset_index()
)
agg_hourly_gdf.columns = ["TRIP_HOUR", "SERVICE_PERIOD", "Total_Boardings"]
agg_hourly = agg_hourly_gdf.to_pandas()
print(f"Agg 4: Hourly pattern done (GPU)")

# --- Agg 5: Route type comparison ---
agg_route_type_gdf = (
    gdf.groupby("ROUTE_TYPE")
    .agg({"BOARDINGS": "sum", "AVG_BOARDINGS": "mean",
          "PEAK_LOAD": "mean", "ROUTE_NUMBER": "nunique", "STOP_ID": "nunique"})
    .reset_index()
    .sort_values("BOARDINGS", ascending=False)
)
agg_route_type_gdf.columns = ["ROUTE_TYPE", "Total_Boardings", "Avg_Boardings",
                               "Avg_Peak_Load", "Num_Routes", "Num_Stops"]
agg_route_type = agg_route_type_gdf.to_pandas()
print(f"Agg 5: Route type comparison done (GPU)")

# --- Agg 6: Time period x Route type ---
agg_time_route_gdf = (
    gdf.groupby(["TIME_PERIOD", "ROUTE_TYPE"])
    .agg({"BOARDINGS": "sum"})
    .reset_index()
)
agg_time_route_gdf.columns = ["TIME_PERIOD", "ROUTE_TYPE", "Total_Boardings"]
agg_time_route = agg_time_route_gdf.to_pandas()
print(f"Agg 6: Time x Route type done (GPU)")

# --- Agg 7: Weekday vs Weekend by Route ---
agg_service_gdf = (
    gdf.groupby(["ROUTE_NAME", "SERVICE_PERIOD"])
    .agg({"BOARDINGS": "sum", "PEAK_LOAD": "mean"})
    .reset_index()
)
agg_service_gdf.columns = ["ROUTE_NAME", "SERVICE_PERIOD",
                            "Total_Boardings", "Avg_Peak_Load"]
agg_service = agg_service_gdf.to_pandas()
print(f"Agg 7: Weekday vs Weekend done (GPU)")

# --- Agg 8: Direction-level ridership ---
agg_direction_gdf = (
    gdf.groupby(["ROUTE_NAME", "DIRECTION_NAME"])
    .agg({"BOARDINGS": "sum", "ALIGHTINGS": "sum"})
    .reset_index()
)
agg_direction_gdf.columns = ["ROUTE_NAME", "DIRECTION_NAME",
                              "Total_Boardings", "Total_Alightings"]
agg_direction = agg_direction_gdf.to_pandas()
print(f"Agg 8: Direction-level done (GPU)")

agg_time_gpu = time.time() - start_agg_gpu
print(f"\n=== All GPU aggregations done in {agg_time_gpu:.2f}s ===")

# Convert full cleaned data to pandas for Dash callbacks
df = gdf.to_pandas()
print(f"Converted to pandas for Dash. Shape: {df.shape}")

Agg 1: Top stops done (GPU)
Agg 2: City ridership done (GPU)
Agg 3: Route ridership done (GPU)
Agg 4: Hourly pattern done (GPU)
Agg 5: Route type comparison done (GPU)
Agg 6: Time x Route type done (GPU)
Agg 7: Weekday vs Weekend done (GPU)
Agg 8: Direction-level done (GPU)

=== All GPU aggregations done in 0.65s ===
Converted to pandas for Dash. Shape: (382721, 37)


## Step 4: CPU vs GPU Benchmark Comparison

This cell runs the same cleaning + aggregation on CPU (pandas) and compares wall-clock times to demonstrate the RAPIDS speedup.

In [5]:
import plotly.graph_objects as go

# --- Run CPU benchmark (same operations as GPU steps above) ---
cpu_df = df_pd.copy()

start_cpu = time.time()

# CPU cleaning
cpu_df = cpu_df.dropna(how="all").dropna(axis=1, how="all")
for col in numeric_cols:
    if col in cpu_df.columns:
        cpu_df[col] = pd.to_numeric(cpu_df[col], errors="coerce")
for col in str_cols:
    if col in cpu_df.columns:
        cpu_df[col] = cpu_df[col].astype(str).str.strip()
cpu_df["TRIP_HOUR"] = (pd.to_numeric(cpu_df["TRIP_TIME"], errors="coerce") * 24).round(0).clip(0, 23)
cpu_df["CITY"] = cpu_df["CITY"].fillna("Unknown")
for col in ["BOARDINGS", "ALIGHTINGS", "AVG_BOARDINGS", "AVG_ALIGHTINGS", "AVG_ACTIVITY"]:
    cpu_df[col] = cpu_df[col].fillna(0)
cpu_df["TOTAL_ACTIVITY"] = cpu_df["BOARDINGS"] + cpu_df["ALIGHTINGS"]
cpu_df = cpu_df.drop_duplicates()

# CPU aggregations
_ = cpu_df.groupby(["STOP_ID", "MAIN_CROSS_STREET", "CITY"]).agg(
    {"BOARDINGS": "sum", "ALIGHTINGS": "sum", "TOTAL_ACTIVITY": "sum", "PEAK_LOAD": "mean"}).reset_index()
_ = cpu_df.groupby("CITY").agg(
    {"BOARDINGS": "sum", "ALIGHTINGS": "sum", "STOP_ID": "nunique", "ROUTE_NUMBER": "nunique"}).reset_index()
_ = cpu_df.groupby(["ROUTE_NUMBER", "ROUTE_NAME"]).agg(
    {"BOARDINGS": "sum", "ALIGHTINGS": "sum", "PEAK_LOAD": "mean", "TRIPS": "sum"}).reset_index()
_ = cpu_df.groupby(["TRIP_HOUR", "SERVICE_PERIOD"]).agg({"BOARDINGS": "sum"}).reset_index()
_ = cpu_df.groupby(["ROUTE_NAME", "SERVICE_PERIOD"]).agg({"BOARDINGS": "sum"}).reset_index()

cpu_total = time.time() - start_cpu
gpu_total = clean_time_gpu + agg_time_gpu

print(f"CPU total (clean + aggregate): {cpu_total:.2f}s")
print(f"GPU total (clean + aggregate): {gpu_total:.2f}s")
print(f"Speedup: {cpu_total / gpu_total:.1f}x faster on GPU")

# --- Benchmark Bar Chart ---
fig_bench = go.Figure(data=[
    go.Bar(name="CPU (pandas)", x=["Data Cleaning + Aggregation"], y=[cpu_total],
           marker_color="#ff7f0e", text=[f"{cpu_total:.2f}s"], textposition="auto"),
    go.Bar(name="GPU (cuDF/RAPIDS)", x=["Data Cleaning + Aggregation"], y=[gpu_total],
           marker_color="#2ca02c", text=[f"{gpu_total:.2f}s"], textposition="auto"),
])
fig_bench.update_layout(
    title=f"CPU vs GPU Performance — {cpu_total/gpu_total:.1f}x Speedup with RAPIDS",
    yaxis_title="Time (seconds)", barmode="group", height=400,
)
fig_bench.show()

CPU total (clean + aggregate): 11.32s
GPU total (clean + aggregate): 1.93s
Speedup: 5.9x faster on GPU


## Step 5: Build & Launch the Dashboard (GPU-processed data, same interactivity as CPU)

**Interactive Components (3):** City Dropdown, Route Type Checklist, Service Period Checklist  
**Callbacks (4):** KPI update, Tab 1 figures, Tab 2 figures, Tab 3 figures  
**Extra Tab 4:** GPU vs CPU Benchmark comparison

In [6]:
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc

# ============================================================
# COLOR PALETTE & CONSTANTS
# ============================================================
COLORS = {
    "Frequent": "#1f77b4", "Local": "#ff7f0e", "Express": "#2ca02c",
    "Light Rail": "#d62728", "Rapid": "#9467bd", "Other": "#8c564b",
    "Unknown": "#7f7f7f",
}
TIME_ORDER = ["Late Night (11-5)", "AM Peak (5-9)", "Midday (9-3)",
              "PM Peak (3-7)", "Evening (7-11)"]

ALL_ROUTE_TYPES = sorted(df["ROUTE_TYPE"].unique().tolist())
ALL_SERVICE_PERIODS = sorted(df["SERVICE_PERIOD"].dropna().unique().tolist())

# ============================================================
# HELPER FUNCTIONS — Build each figure from filtered data
# ============================================================

def build_fig_top_stops(filtered, title_suffix="All Cities"):
    stops_f = (
        filtered.groupby(["STOP_ID", "MAIN_CROSS_STREET", "CITY"])
        .agg(Total_Boardings=("BOARDINGS", "sum"),
             Total_Alightings=("ALIGHTINGS", "sum"),
             Avg_Peak_Load=("PEAK_LOAD", "mean"))
        .reset_index().sort_values("Total_Boardings", ascending=False).head(20)
    )
    fig = px.bar(stops_f.sort_values("Total_Boardings"),
        x="Total_Boardings", y="MAIN_CROSS_STREET", orientation="h", color="CITY",
        title=f"Fig 1: Top 20 Busiest Stops — {title_suffix}",
        labels={"Total_Boardings": "Total Boardings", "MAIN_CROSS_STREET": "Stop"},
        hover_data=["CITY", "Total_Alightings", "Avg_Peak_Load"])
    fig.update_layout(yaxis=dict(tickfont=dict(size=10)), height=550)
    return fig

def build_fig_city(filtered):
    agg = (filtered.groupby("CITY")
        .agg(Total_Boardings=("BOARDINGS", "sum"),
             Num_Stops=("STOP_ID", "nunique"), Num_Routes=("ROUTE_NUMBER", "nunique"))
        .reset_index().sort_values("Total_Boardings", ascending=False).head(15))
    fig = px.treemap(agg, path=["CITY"], values="Total_Boardings",
        color="Total_Boardings", color_continuous_scale="Blues",
        title="Fig 2: Ridership Distribution by City (Top 15)",
        hover_data=["Num_Stops", "Num_Routes"])
    return fig

def build_fig_routes(filtered, title_suffix="All Cities"):
    routes_f = (filtered.groupby(["ROUTE_NUMBER", "ROUTE_NAME", "ROUTE_TYPE"])
        .agg(Total_Boardings=("BOARDINGS", "sum"),
             Avg_Peak_Load=("PEAK_LOAD", "mean"), Num_Trips=("TRIPS", "sum"))
        .reset_index().sort_values("Total_Boardings", ascending=False).head(15))
    fig = px.bar(routes_f.sort_values("Total_Boardings"),
        x="Total_Boardings", y="ROUTE_NAME", orientation="h",
        color="ROUTE_TYPE", color_discrete_map=COLORS,
        title=f"Fig 3: Top 15 Routes — {title_suffix}",
        labels={"Total_Boardings": "Total Boardings", "ROUTE_NAME": "Route"},
        hover_data=["Avg_Peak_Load", "Num_Trips"])
    fig.update_layout(yaxis=dict(tickfont=dict(size=9)), height=500)
    return fig

def build_fig_heatmap(filtered):
    agg = (filtered.groupby(["TRIP_HOUR", "SERVICE_PERIOD"])
        .agg(Total_Boardings=("BOARDINGS", "sum")).reset_index())
    pivot = agg.pivot_table(index="SERVICE_PERIOD", columns="TRIP_HOUR",
        values="Total_Boardings", fill_value=0)
    fig = px.imshow(pivot,
        labels=dict(x="Hour of Day", y="Service Period", color="Total Boardings"),
        title="Fig 4: Ridership Heatmap — Hour of Day vs Service Period",
        color_continuous_scale="YlOrRd", aspect="auto")
    fig.update_layout(height=350)
    return fig

def build_fig_time_route(filtered):
    agg = (filtered.groupby(["TIME_PERIOD", "ROUTE_TYPE"])
        .agg(Total_Boardings=("BOARDINGS", "sum")).reset_index())
    fig = px.bar(agg, x="TIME_PERIOD", y="Total_Boardings",
        color="ROUTE_TYPE", color_discrete_map=COLORS, barmode="group",
        title="Fig 5: Boardings by Time Period & Route Type",
        category_orders={"TIME_PERIOD": TIME_ORDER})
    fig.update_layout(height=400)
    return fig

def build_fig_weekday(filtered):
    agg_r = (filtered.groupby(["ROUTE_NAME"]).agg(Total_Boardings=("BOARDINGS", "sum"))
        .reset_index().sort_values("Total_Boardings", ascending=False))
    top10 = agg_r.head(10)["ROUTE_NAME"].tolist()
    agg_svc = (filtered[filtered["ROUTE_NAME"].isin(top10)]
        .groupby(["ROUTE_NAME", "SERVICE_PERIOD"])
        .agg(Total_Boardings=("BOARDINGS", "sum")).reset_index())
    fig = px.bar(agg_svc, x="ROUTE_NAME", y="Total_Boardings",
        color="SERVICE_PERIOD", barmode="group",
        title="Fig 6: Weekday vs Weekend Ridership — Top 10 Routes",
        labels={"ROUTE_NAME": "Route", "Total_Boardings": "Total Boardings"})
    fig.update_layout(xaxis_tickangle=-45, height=450)
    return fig

def build_fig_bubble(filtered):
    agg = (filtered.groupby("ROUTE_TYPE")
        .agg(Total_Boardings=("BOARDINGS", "sum"), Avg_Peak_Load=("PEAK_LOAD", "mean"),
             Num_Routes=("ROUTE_NUMBER", "nunique"), Num_Stops=("STOP_ID", "nunique"))
        .reset_index().sort_values("Total_Boardings", ascending=False))
    fig = px.scatter(agg, x="Num_Routes", y="Total_Boardings",
        size="Avg_Peak_Load", color="ROUTE_TYPE", color_discrete_map=COLORS,
        title="Fig 7: Route Type Performance (bubble = Avg Peak Load)",
        labels={"Num_Routes": "Number of Routes", "Total_Boardings": "Total Boardings"},
        hover_data=["Num_Stops", "Avg_Peak_Load"])
    fig.update_layout(height=450)
    return fig

def build_fig_balance(filtered):
    agg = (filtered.groupby(["ROUTE_NUMBER", "ROUTE_NAME", "ROUTE_TYPE"])
        .agg(Total_Boardings=("BOARDINGS", "sum"), Total_Alightings=("ALIGHTINGS", "sum"))
        .reset_index())
    fig = px.scatter(agg, x="Total_Boardings", y="Total_Alightings",
        color="ROUTE_TYPE", color_discrete_map=COLORS, hover_name="ROUTE_NAME",
        title="Fig 8: Boarding vs Alighting Balance by Route",
        labels={"Total_Boardings": "Total Boardings", "Total_Alightings": "Total Alightings"})
    if len(agg) > 0:
        max_val = max(agg["Total_Boardings"].max(), agg["Total_Alightings"].max())
        fig.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val],
            mode="lines", line=dict(dash="dash", color="gray"), showlegend=False))
    fig.update_layout(height=450)
    return fig

def build_fig_load(filtered):
    agg = (filtered.groupby(["ROUTE_NUMBER", "ROUTE_NAME", "ROUTE_TYPE"])
        .agg(Avg_Peak_Load=("PEAK_LOAD", "mean")).reset_index().nlargest(20, "Avg_Peak_Load"))
    fig = px.bar(agg.sort_values("Avg_Peak_Load"),
        x="Avg_Peak_Load", y="ROUTE_NAME", orientation="h",
        color="ROUTE_TYPE", color_discrete_map=COLORS,
        title="Fig 9: Top 20 Most Crowded Routes (Avg Peak Load)",
        labels={"Avg_Peak_Load": "Avg Peak Load (passengers)", "ROUTE_NAME": "Route"})
    fig.update_layout(yaxis=dict(tickfont=dict(size=9)), height=550)
    return fig

print("All 9 figure-builder functions defined (GPU notebook).")

All 9 figure-builder functions defined (GPU notebook).


In [7]:
# ============================================================
# DASH APP — 3 Interactive Components, 4 Callbacks, 9 Visuals + Benchmark Tab
# ============================================================

app = Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])

# --- Filter Options ---
city_options = [{"label": "All Cities", "value": "ALL"}] + [
    {"label": c, "value": c}
    for c in sorted(df["CITY"].dropna().unique()) if c != "Unknown"
]

# --- Global Filters Bar (above tabs) ---
filters_bar = dbc.Card(
    dbc.CardBody([
        dbc.Row([
            dbc.Col([
                html.Label("Filter by City:", className="fw-bold"),
                dcc.Dropdown(id="city-filter", options=city_options,
                             value="ALL", clearable=False),
            ], md=3),
            dbc.Col([
                html.Label("Filter by Route Type:", className="fw-bold"),
                dcc.Checklist(
                    id="route-type-filter",
                    options=[{"label": f"  {rt}", "value": rt} for rt in ALL_ROUTE_TYPES],
                    value=ALL_ROUTE_TYPES, inline=True,
                    inputStyle={"marginRight": "4px", "marginLeft": "12px"},
                ),
            ], md=9),
        ]),
    ]),
    className="mb-3 shadow-sm",
)

# --- KPI Cards (dynamic — updated by Callback 1) ---
kpi_row = dbc.Row([
    dbc.Col(dbc.Card(dbc.CardBody([
        html.H6("Total Boardings", className="card-title text-muted", style={"fontSize": "0.85rem"}),
        html.H4(id="kpi-boardings", className="text-primary fw-bold"),
    ]), className="shadow-sm"), md=3),
    dbc.Col(dbc.Card(dbc.CardBody([
        html.H6("Unique Stops", className="card-title text-muted", style={"fontSize": "0.85rem"}),
        html.H4(id="kpi-stops", className="text-success fw-bold"),
    ]), className="shadow-sm"), md=3),
    dbc.Col(dbc.Card(dbc.CardBody([
        html.H6("Routes Served", className="card-title text-muted", style={"fontSize": "0.85rem"}),
        html.H4(id="kpi-routes", className="text-info fw-bold"),
    ]), className="shadow-sm"), md=3),
    dbc.Col(dbc.Card(dbc.CardBody([
        html.H6("Cities Covered", className="card-title text-muted", style={"fontSize": "0.85rem"}),
        html.H4(id="kpi-cities", className="text-warning fw-bold"),
    ]), className="shadow-sm"), md=3),
], className="mb-3")

# --- Tab 1: WHERE ---
tab1 = dbc.Tab(label="WHERE: Stops & Cities", children=[
    dbc.Container([
        html.Br(),
        html.P("Which stops, routes, and cities generate the most ridership?",
               className="text-muted fst-italic"),
        dbc.Row([
            dbc.Col(dcc.Graph(id="fig-top-stops"), md=7),
            dbc.Col(dcc.Graph(id="fig-city"), md=5),
        ]),
        dbc.Row([dbc.Col(dcc.Graph(id="fig-routes"), md=12)]),
    ], fluid=True),
])

# --- Tab 2: WHEN (has its own Service Period filter) ---
tab2 = dbc.Tab(label="WHEN: Time Patterns", children=[
    dbc.Container([
        html.Br(),
        html.P("When do riders use VTA the most — by hour, time period, and day type?",
               className="text-muted fst-italic"),
        dbc.Row([
            dbc.Col([
                html.Label("Filter by Service Period:", className="fw-bold"),
                dcc.Checklist(
                    id="service-period-filter",
                    options=[{"label": f"  {sp}", "value": sp} for sp in ALL_SERVICE_PERIODS],
                    value=ALL_SERVICE_PERIODS, inline=True,
                    inputStyle={"marginRight": "4px", "marginLeft": "12px"},
                ),
            ], md=12),
        ], className="mb-3"),
        dbc.Row([dbc.Col(dcc.Graph(id="fig-heatmap"), md=12)]),
        dbc.Row([
            dbc.Col(dcc.Graph(id="fig-time-route"), md=6),
            dbc.Col(dcc.Graph(id="fig-weekday"), md=6),
        ]),
    ], fluid=True),
])

# --- Tab 3: HOW ---
tab3 = dbc.Tab(label="HOW: Performance & Crowding", children=[
    dbc.Container([
        html.Br(),
        html.P("How do different route types perform, and where is overcrowding or underutilization?",
               className="text-muted fst-italic"),
        dbc.Row([
            dbc.Col(dcc.Graph(id="fig-bubble"), md=6),
            dbc.Col(dcc.Graph(id="fig-balance"), md=6),
        ]),
        dbc.Row([dbc.Col(dcc.Graph(id="fig-load"), md=12)]),
    ], fluid=True),
])

# --- Tab 4: GPU BENCHMARK ---
tab4 = dbc.Tab(label="GPU vs CPU Benchmark", children=[
    dbc.Container([
        html.Br(),
        html.H5("NVIDIA RAPIDS Acceleration Results"),
        html.P(f"Dataset: 388K+ rows, 32 columns | "
               f"CPU time: {cpu_total:.2f}s | GPU time: {gpu_total:.2f}s | "
               f"Speedup: {cpu_total/gpu_total:.1f}x",
               className="text-muted"),
        dbc.Row([dbc.Col(dcc.Graph(figure=fig_bench), md=8)]),
        html.Hr(),
        dbc.Alert([
            html.H6("Why GPU acceleration matters:", className="alert-heading"),
            html.Ul([
                html.Li("cuDF provides pandas-like API but runs on NVIDIA GPU"),
                html.Li("Cleaning & aggregation of 388K rows is parallelized across GPU cores"),
                html.Li("Enables real-time interactive dashboards on larger datasets"),
                html.Li("Same code logic, just swap pd → cudf for GPU execution"),
            ]),
        ], color="success"),
    ], fluid=True),
])

# --- Full Layout ---
app.layout = dbc.Container([
    html.H2("VTA 2025 Ridership Dashboard (GPU-Accelerated)",
            className="text-center mt-3 mb-2 fw-bold"),
    html.P("Optimizing Silicon Valley Transit: Where, When & How People Ride",
           className="text-center text-muted mb-1"),
    html.P("Powered by NVIDIA RAPIDS cuDF",
           className="text-center text-success mb-3", style={"fontSize": "0.9rem"}),
    filters_bar,
    kpi_row,
    html.Hr(),
    dbc.Tabs([tab1, tab2, tab3, tab4]),
    html.Hr(),
    html.Footer(
        html.Small("Data: VTA Oct 2025 Ridership by Stop & Station | "
                   "Dashboard: Plotly Dash + NVIDIA RAPIDS (GPU)", className="text-muted"),
        className="text-center mb-3"
    ),
], fluid=True)


# ============================================================
# HELPER: Apply global filters (City + Route Type)
# ============================================================
def apply_global_filters(city, route_types):
    filtered = df[df["ROUTE_TYPE"].isin(route_types)]
    if city != "ALL":
        filtered = filtered[filtered["CITY"] == city]
    return filtered


# ============================================================
# CALLBACK 1: KPI Cards (City + Route Type → 4 KPI text values)
# ============================================================
@app.callback(
    [Output("kpi-boardings", "children"),
     Output("kpi-stops", "children"),
     Output("kpi-routes", "children"),
     Output("kpi-cities", "children")],
    [Input("city-filter", "value"),
     Input("route-type-filter", "value")]
)
def update_kpis(city, route_types):
    filtered = apply_global_filters(city, route_types)
    return (f"{int(filtered['BOARDINGS'].sum()):,}",
            f"{filtered['STOP_ID'].nunique():,}",
            f"{filtered['ROUTE_NUMBER'].nunique():,}",
            f"{filtered['CITY'].nunique():,}")


# ============================================================
# CALLBACK 2: Tab 1 — WHERE (City + Route Type → Fig 1, 2, 3)
# ============================================================
@app.callback(
    [Output("fig-top-stops", "figure"),
     Output("fig-city", "figure"),
     Output("fig-routes", "figure")],
    [Input("city-filter", "value"),
     Input("route-type-filter", "value")]
)
def update_tab1(city, route_types):
    filtered = apply_global_filters(city, route_types)
    title_suffix = city if city != "ALL" else "All Cities"
    return (build_fig_top_stops(filtered, title_suffix),
            build_fig_city(filtered),
            build_fig_routes(filtered, title_suffix))


# ============================================================
# CALLBACK 3: Tab 2 — WHEN (Route Type + Service Period → Fig 4, 5, 6)
# ============================================================
@app.callback(
    [Output("fig-heatmap", "figure"),
     Output("fig-time-route", "figure"),
     Output("fig-weekday", "figure")],
    [Input("route-type-filter", "value"),
     Input("service-period-filter", "value")]
)
def update_tab2(route_types, service_periods):
    filtered = df[
        df["ROUTE_TYPE"].isin(route_types) &
        df["SERVICE_PERIOD"].isin(service_periods)
    ]
    return (build_fig_heatmap(filtered),
            build_fig_time_route(filtered),
            build_fig_weekday(filtered))


# ============================================================
# CALLBACK 4: Tab 3 — HOW (City + Route Type → Fig 7, 8, 9)
# ============================================================
@app.callback(
    [Output("fig-bubble", "figure"),
     Output("fig-balance", "figure"),
     Output("fig-load", "figure")],
    [Input("city-filter", "value"),
     Input("route-type-filter", "value")]
)
def update_tab3(city, route_types):
    filtered = apply_global_filters(city, route_types)
    return (build_fig_bubble(filtered),
            build_fig_balance(filtered),
            build_fig_load(filtered))


print("Dashboard ready! 3 interactive components + 4 callbacks + 9 visuals + 4 dynamic KPIs + benchmark tab")

Dashboard ready! 3 interactive components + 4 callbacks + 9 visuals + 4 dynamic KPIs + benchmark tab


In [8]:
# ============================================================
# RUN THE DASHBOARD (Google Colab — opens in new browser tab)
# ============================================================
import threading
from google.colab import output

# Run Dash server in a background thread
def run_dash():
    app.run(host="0.0.0.0", port=8051, debug=False, use_reloader=False)

threading.Thread(target=run_dash, daemon=True).start()

import time
time.sleep(2)  # Wait for server to start

# Open the dashboard in a new browser tab via Colab's proxy
output.serve_kernel_port_as_window(8051)
print("Dashboard opened in a new browser tab!")

Dash is running on http://0.0.0.0:8051/



INFO:dash.dash:Dash is running on http://0.0.0.0:8051/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8051
 * Running on http://172.28.0.12:8051
INFO:werkzeug:Press CTRL+C to quit


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

Dashboard opened in a new browser tab!
